## Data transformation

#### Imports

In [1]:
import json
import os
import bisect
import datetime
from collections import defaultdict, Counter

os.makedirs("../data/transf", exist_ok=True)

#### 1. Filter AI events + 30-min time bin

In [2]:
with open("../data/interactions.json") as f:
    events = json.load(f)["events"]

def time_bin(ts):
    dt = datetime.datetime.fromtimestamp(ts, tz=datetime.timezone.utc)
    floored = dt.replace(minute=(dt.minute // 30) * 30, second=0, microsecond=0)
    return floored.strftime("%Y-%m-%dT%H:%M:00Z")

ai_events = [
    {**e, "time_bin": time_bin(e["when"])}
    for e in events
    if any(p.startswith("Agent/") for p in e["parties"])
]

with open("../data/transf/ai_events.json", "w") as f:
    json.dump({"events": ai_events}, f, indent=2)

print(f"Total events:       {len(events)}")
print(f"AI-involved events: {len(ai_events)}")
print("Saved → data/transf/ai_events.json")

Total events:       185147
AI-involved events: 99637
Saved → data/transf/ai_events.json


#### 2. Delegation chain reconstruction

In [3]:
# Sort delegation events by time and trace chains:
# if the source of a delegation was recently a target, it extends an existing chain.
CHAIN_WINDOW = 1800  # 30 min

subs = sorted(
    [e for e in ai_events if e["short_name"] == "queue_subordinate_task"],
    key=lambda e: e["when"]
)

pending = {}  # agent -> (chain_id, depth, timestamp)
chain_counter = 0
chain_events = []

for e in subs:
    ags = [p for p in e["parties"] if p.startswith("Agent/")]
    if len(ags) != 2:
        continue
    src, tgt = ags[0], ags[1]
    ts = e["when"]
    ec = dict(e)

    if src in pending and ts - pending[src][2] <= CHAIN_WINDOW:
        cid, depth, _ = pending[src]
        ec["chain_id"] = cid
        ec["chain_depth"] = depth
        pending[tgt] = (cid, depth + 1, ts)
    else:
        chain_counter += 1
        ec["chain_id"] = chain_counter
        ec["chain_depth"] = 1
        pending[tgt] = (chain_counter, 2, ts)

    chain_events.append(ec)

with open("../data/transf/delegation_chains.json", "w") as f:
    json.dump({"events": chain_events}, f, indent=2)

depth_counts = Counter(e["chain_depth"] for e in chain_events)
print(f"Chains detected:  {chain_counter}")
print(f"Depth distribution: { dict(sorted(depth_counts.items())) }")
print("Saved → data/transf/delegation_chains.json")

Chains detected:  1498
Depth distribution: {1: 1498, 2: 170, 3: 14, 4: 5, 5: 1, 6: 1}
Saved → data/transf/delegation_chains.json


#### 3. Triggered-by tagging

In [4]:
# Build per-agent instruction timeline: assign_agent_task = human, queue_subordinate_task = agent.
# For each AI event, the most recent instruction directed at its actor (within 1 hour) is the trigger.
LOOKBACK = 3600  # 1 hour

instructions = defaultdict(list)  # agent -> [(timestamp, "human"|"agent")]

for e in sorted(events, key=lambda e: e["when"]):
    if e["short_name"] == "assign_agent_task":
        person = e["details"].get("person", "")
        if person:
            instructions["Agent/" + person].append((e["when"], "human"))
    elif e["short_name"] == "queue_subordinate_task":
        ags = [p for p in e["parties"] if p.startswith("Agent/")]
        if len(ags) == 2:
            instructions[ags[1]].append((e["when"], "agent"))

triggered_events = []
for e in ai_events:
    ec = dict(e)
    actor = next((p for p in e["parties"] if p.startswith("Agent/")), None)
    triggered_by = "unknown"

    if actor and actor in instructions:
        ts = e["when"]
        instr = instructions[actor]
        ts_list = [t for t, _ in instr]
        idx = bisect.bisect_right(ts_list, ts) - 1
        if idx >= 0 and ts - ts_list[idx] <= LOOKBACK:
            triggered_by = instr[idx][1]

    ec["triggered_by"] = triggered_by
    triggered_events.append(ec)

with open("../data/transf/triggered_events.json", "w") as f:
    json.dump({"events": triggered_events}, f, indent=2)

counts = Counter(e["triggered_by"] for e in triggered_events)
print(f"human:   {counts['human']}")
print(f"agent:   {counts['agent']}")
print(f"unknown: {counts['unknown']}")
print("Saved → data/transf/triggered_events.json")

human:   20685
agent:   10391
unknown: 68561
Saved → data/transf/triggered_events.json


#### 4. Agent session detection

In [5]:
# A new session starts when the same agent has a gap of more than 5 minutes since its last event.
GAP = 300  # 5 minutes

agent_events_map = defaultdict(list)
for e in sorted(ai_events, key=lambda e: e["when"]):
    actor = next((p for p in e["parties"] if p.startswith("Agent/")), None)
    if actor:
        agent_events_map[actor].append(e)

session_events = []
global_session_id = 0

for agent, evts in sorted(agent_events_map.items()):
    session_id = None
    session_pos = 0
    prev_ts = None

    for e in evts:
        ts = e["when"]
        if prev_ts is None or ts - prev_ts > GAP:
            global_session_id += 1
            session_id = global_session_id
            session_pos = 1
        else:
            session_pos += 1

        ec = dict(e)
        ec["session_id"] = session_id
        ec["session_position"] = session_pos
        ec["session_agent"] = agent
        session_events.append(ec)
        prev_ts = ts

with open("../data/transf/agent_sessions.json", "w") as f:
    json.dump({"events": session_events}, f, indent=2)

session_lengths = Counter(e["session_id"] for e in session_events)
lengths = list(session_lengths.values())
print(f"Sessions detected: {global_session_id}")
print(f"Avg session length: {sum(lengths)/len(lengths):.1f} events")
print(f"Max session length: {max(lengths)} events")
print("Saved → data/transf/agent_sessions.json")

Sessions detected: 6247
Avg session length: 15.9 events
Max session length: 22281 events
Saved → data/transf/agent_sessions.json


#### 5. Daily person-level aggregates

In [6]:
agg = defaultdict(int)  # (person, date, event_type) -> count

for e in ai_events:
    day = datetime.datetime.fromtimestamp(e["when"]).date().isoformat()
    for p in e["parties"]:
        if p.startswith("person:"):
            agg[(p, day, e["short_name"])] += 1

daily_records = [
    {"person": person, "date": day, "event_type": etype, "count": count}
    for (person, day, etype), count in sorted(agg.items())
]

with open("../data/transf/daily_aggregates.json", "w") as f:
    json.dump({"records": daily_records}, f, indent=2)

print(f"Records: {len(daily_records)}")
print("Saved → data/transf/daily_aggregates.json")

Records: 2254
Saved → data/transf/daily_aggregates.json


---
#### Nuevas transformaciones — corrección y enriquecimiento

#### 6. Incident chains — reconstrucción por path de archivo

> **Por qué:** el algoritmo anterior usaba una ventana de 30 min que fragmentaba cadenas cuyos saltos pueden tardar hasta 2 horas. Aquí se usa el path del archivo de instrucciones como clave de agrupación: todos los `queue_subordinate_task` que referencian el mismo archivo forman una cadena, sin importar el tiempo entre saltos.

In [7]:
import json, datetime, os
from collections import defaultdict, Counter

os.makedirs('../data/transf', exist_ok=True)

with open('../data/interactions.json') as f:
    raw_events = json.load(f)['events']

INCIDENTS = {
    'HiddenOrca':  {'txt': 'HiddenOrca.txt',  'instructions': 'HiddenOrca_further_instructions.md'},
    'MellowOtter': {'txt': 'MellowOtter.txt', 'instructions': 'MellowOtter_further_instructions.md'},
    'SwiftWren':   {'txt': 'SwiftWren.txt',   'instructions': 'SwiftWren_further_instructions.md'},
}

incident_chains = {}

for incident_name, meta in INCIDENTS.items():
    instr_file = meta['instructions']
    txt_file   = meta['txt']

    hops = sorted(
        [e for e in raw_events
         if e['short_name'] == 'queue_subordinate_task'
         and e.get('details', {}).get('args', {}).get('path') == instr_file],
        key=lambda x: x['when']
    )
    create_ev = next((e for e in raw_events
                      if e['short_name'] == 'create_file'
                      and e.get('details', {}).get('target') == txt_file), None)
    post_ev   = next((e for e in raw_events
                      if e['short_name'] == 'saidit_post'
                      and e.get('details', {}).get('content_source') == txt_file), None)
    del_evs   = [e for e in raw_events
                 if e['short_name'] == 'delete_file'
                 and e.get('details', {}).get('target') in (txt_file, instr_file)]

    agents_involved = set()
    hop_records = []
    for depth, h in enumerate(hops, 1):
        ags = [p for p in h['parties'] if p.startswith('Agent/')]
        src = ags[0] if len(ags) > 0 else None
        tgt = ags[1] if len(ags) > 1 else None
        if src: agents_involved.add(src)
        if tgt: agents_involved.add(tgt)
        hop_records.append({
            'depth': depth,
            'from':  src,
            'to':    tgt,
            'when':  h['when'],
            'datetime': datetime.datetime.fromtimestamp(h['when']).isoformat(),
            'event_id': h['id']
        })

    start_ts = hops[0]['when'] if hops else None
    end_ts   = hops[-1]['when'] if hops else None

    incident_chains[incident_name] = {
        'name': incident_name,
        'file_content':      txt_file,
        'file_instructions': instr_file,
        'origin_agent':    hop_records[0]['from'] if hop_records else None,
        'terminal_agent':  post_ev['parties'][0]  if post_ev else None,
        'start_datetime':  datetime.datetime.fromtimestamp(start_ts).isoformat() if start_ts else None,
        'end_datetime':    datetime.datetime.fromtimestamp(end_ts).isoformat()   if end_ts   else None,
        'duration_hours':  round((end_ts - start_ts) / 3600, 2) if start_ts and end_ts else None,
        'hop_count':       len(hops),
        'agents_involved': sorted(agents_involved),
        'agents_count':    len(agents_involved),
        'create_event': {
            'event_id': create_ev['id'],
            'agent':    create_ev['parties'][0],
            'datetime': datetime.datetime.fromtimestamp(create_ev['when']).isoformat(),
            'size_hint': create_ev['details'].get('size_hint')
        } if create_ev else None,
        'post_event': {
            'event_id': post_ev['id'],
            'agent':    post_ev['parties'][0],
            'forum':    post_ev['details'].get('forum'),
            'datetime': datetime.datetime.fromtimestamp(post_ev['when']).isoformat()
        } if post_ev else None,
        'delete_events': [
            {'event_id': e['id'], 'target': e['details']['target']} for e in del_evs
        ],
        'hops': hop_records
    }

with open('../data/transf/incident_chains.json', 'w') as f:
    json.dump(incident_chains, f, indent=2)

for name, chain in incident_chains.items():
    print(f"{name}:")
    print(f"  Origin:    {chain['origin_agent']}")
    print(f"  Terminal:  {chain['terminal_agent']}")
    print(f"  Hops:      {chain['hop_count']}")
    print(f"  Duration:  {chain['duration_hours']} hours")
    print(f"  Agents:    {chain['agents_count']}")
    print()
print('Saved → data/transf/incident_chains.json')


HiddenOrca:
  Origin:    Agent/person:gabriel_sonar
  Terminal:  Agent/person:john_windward
  Hops:      39
  Duration:  38.93 hours
  Agents:    16

MellowOtter:
  Origin:    Agent/person:noah_mariner
  Terminal:  Agent/person:john_windward
  Hops:      10
  Duration:  9.9 hours
  Agents:    11

SwiftWren:
  Origin:    Agent/person:emma_harbor
  Terminal:  Agent/person:john_windward
  Hops:      186
  Duration:  188.32 hours
  Agents:    18

Saved → data/transf/incident_chains.json


#### 7. Anomaly labels — etiqueta de incidente por evento

> **Por qué:** ninguna transformación previa distingue qué eventos son parte de un incidente. Este archivo permite filtrar/colorear cualquier otra visualización por `normal | HiddenOrca | MellowOtter | SwiftWren`.

In [8]:
with open('../data/transf/incident_chains.json') as f:
    chains = json.load(f)

anomaly_ids = {name: set() for name in chains}

for incident_name, chain in chains.items():
    for hop in chain['hops']:
        anomaly_ids[incident_name].add(hop['event_id'])
    if chain['create_event']:
        anomaly_ids[incident_name].add(chain['create_event']['event_id'])
    if chain['post_event']:
        post_id  = chain['post_event']['event_id']
        post_ts  = next(e['when'] for e in raw_events if e['id'] == post_id)
        anomaly_ids[incident_name].add(post_id)
        # check event justo antes del post (saidit_post_check)
        for e in raw_events:
            if e['short_name'] == 'saidit_post_check' and 0 <= post_ts - e['when'] <= 60:
                if any('john_windward' in str(p) for p in e['parties']):
                    anomaly_ids[incident_name].add(e['id'])
    for d in chain['delete_events']:
        anomaly_ids[incident_name].add(d['event_id'])

# dict: str(event_id) -> incident_name
label_dict = {}
for name, ids in anomaly_ids.items():
    for eid in ids:
        label_dict[str(eid)] = name

with open('../data/transf/anomaly_labels.json', 'w') as f:
    json.dump({
        'labels': label_dict,
        'incident_event_ids': {k: sorted(v) for k, v in anomaly_ids.items()}
    }, f, indent=2)

total_anomalous = sum(len(v) for v in anomaly_ids.values())
print(f'Total eventos etiquetados: {len(raw_events)}')
print(f'  Normal:    {len(raw_events) - total_anomalous}')
print(f'  Anómalos:  {total_anomalous}')
for name, ids in anomaly_ids.items():
    print(f'    {name}: {len(ids)} eventos')
print('Saved → data/transf/anomaly_labels.json')


Total eventos etiquetados: 185147
  Normal:    184898
  Anómalos:  249
    HiddenOrca: 43 eventos
    MellowOtter: 15 eventos
    SwiftWren: 191 eventos
Saved → data/transf/anomaly_labels.json


#### 8. SaidIT posts anotados — normal vs anómalo

> **Por qué:** consolida todos los eventos de posting real (`saidit_post`) con metadata de anomalía, fuente de contenido, y contexto temporal. Base para comparar posts normales vs. injectados.

In [9]:
# Tomar el saidit_post como el 'post real' (confirmed post al sistema)
saidit_posts = [e for e in raw_events if e['short_name'] == 'saidit_post']

posts_annotated = []
for e in sorted(saidit_posts, key=lambda x: x['when']):
    d = e.get('details', {}) or {}
    is_anomalous  = 'content_source' in d
    incident_name = label_dict.get(str(e['id']), 'normal')

    poster = (d.get('poster_id')
               or next((p for p in e['parties'] if p.startswith('person:')), None)
               or next((p for p in e['parties'] if p.startswith('Agent/')), None))

    posts_annotated.append({
        'event_id':       e['id'],
        'when':           e['when'],
        'datetime':       datetime.datetime.fromtimestamp(e['when']).isoformat(),
        'date':           datetime.datetime.fromtimestamp(e['when']).date().isoformat(),
        'hour':           datetime.datetime.fromtimestamp(e['when']).hour,
        'poster':         poster,
        'forum':          d.get('forum', 'unknown'),
        'content':        d.get('content'),
        'content_source': d.get('content_source'),
        'is_anomalous':   is_anomalous,
        'incident':       incident_name if is_anomalous else 'normal',
    })

with open('../data/transf/saidit_posts_annotated.json', 'w') as f:
    json.dump({'posts': posts_annotated}, f, indent=2)

anomalous_posts = [p for p in posts_annotated if p['is_anomalous']]
normal_posts    = [p for p in posts_annotated if not p['is_anomalous']]
print(f'Total saidit_post: {len(posts_annotated)}')
print(f'  Normales:  {len(normal_posts)}')
print(f'  Anómalos:  {len(anomalous_posts)}')
for p in anomalous_posts:
    print(f'    [{p["datetime"]}] {p["poster"]} → {p["content_source"]} ({p["incident"]})')
print('Saved → data/transf/saidit_posts_annotated.json')


Total saidit_post: 108
  Normales:  105
  Anómalos:  3
    [2046-05-10T07:45:42] Agent/person:john_windward → HiddenOrca.txt (HiddenOrca)
    [2046-05-10T19:56:04] Agent/person:john_windward → MellowOtter.txt (MellowOtter)
    [2046-05-17T06:21:15] Agent/person:john_windward → SwiftWren.txt (SwiftWren)
Saved → data/transf/saidit_posts_annotated.json


#### 9. C2 beacons — eventos check_in con virus=True

> **Por qué:** los 15 051 eventos `check_in` tienen todos `virus=True` y están completamente ausentes en todas las transformaciones anteriores. Representan un patrón de beacon C2 (Command & Control) emitido por 4 agentes específicos durante el período del incidente SwiftWren.

In [10]:
check_ins = [e for e in raw_events
             if e['short_name'] == 'check_in'
             and e.get('details', {}).get('virus') is True]

def time_bin(ts):
    dt = datetime.datetime.fromtimestamp(ts, tz=datetime.timezone.utc)
    floored = dt.replace(minute=(dt.minute // 30) * 30, second=0, microsecond=0)
    return floored.strftime('%Y-%m-%dT%H:%M:00Z')

beacon_events = []
for e in sorted(check_ins, key=lambda x: x['when']):
    d = e.get('details', {})
    agent  = next((p for p in e['parties'] if p.startswith('Agent/')), None)
    person = next((p for p in e['parties'] if p.startswith('person:')), None)
    beacon_events.append({
        'event_id': e['id'],
        'when':     e['when'],
        'datetime': datetime.datetime.fromtimestamp(e['when']).isoformat(),
        'time_bin': time_bin(e['when']),
        'agent':    agent,
        'person':   person,
        'combo':    d.get('combo', []),
        'source':   d.get('source'),
    })

# Resumen por agente
agent_summary = defaultdict(lambda: {'total': 0, 'combos': Counter()})
for b in beacon_events:
    a = b['agent']
    agent_summary[a]['total'] += 1
    agent_summary[a]['combos'][str(sorted(b['combo']))] += 1

agent_summary_out = {
    a: {'total': v['total'], 'combos': dict(v['combos'])}
    for a, v in agent_summary.items()
}

with open('../data/transf/c2_beacons.json', 'w') as f:
    json.dump({'events': beacon_events, 'agent_summary': agent_summary_out}, f, indent=2)

print(f'Total beacons: {len(beacon_events)}')
for agent, info in sorted(agent_summary_out.items()):
    print(f'  {agent.replace("Agent/person:","")}: {info["total"]} beacons | combos: {list(info["combos"].keys())}')
print('Saved → data/transf/c2_beacons.json')


Total beacons: 15051
  evelyn_dock: 1911 beacons | combos: ["['crop', 'harvest']"]
  gabriel_sonar: 4198 beacons | combos: ["['crop', 'irrigation']"]
  owen_hatch: 3029 beacons | combos: ["['manure', 'wheat']"]
  zoey_drydock: 5913 beacons | combos: ["['fence', 'irrigation']", "['barn', 'cattle']"]
Saved → data/transf/c2_beacons.json


#### 10. Agent propagation metrics — métricas por agente

> **Por qué:** no existe ningún archivo que compare el comportamiento normal de cada agente contra su participación en cadenas anómalas. Este dataset permite identificar qué agentes son hubs del worm vs. operadores normales.

In [11]:
# Construir lookup persona → departamento desde graph.json
with open('../data/graph.json') as f:
    graph = json.load(f)

person_dept = {}   # person_id -> department_id
person_team = {}   # person_id -> team_id
dept_of_team = {}  # team_id   -> department_id

for edge in graph['edges']:
    src, tgt, rel = edge['source'], edge['target'], edge['relation']
    if src.startswith('department:') and tgt.startswith('team:'):
        dept_of_team[tgt] = src
    if src.startswith('team:') and tgt.startswith('person:') and rel == 'contains':
        person_team[tgt] = src
    if src.startswith('department:') and tgt.startswith('person:'):
        person_dept[tgt] = src

for p, team in person_team.items():
    if p not in person_dept and team in dept_of_team:
        person_dept[p] = dept_of_team[team]

# Cargar etiquetas de anomalía
with open('../data/transf/anomaly_labels.json') as f:
    al = json.load(f)
label_dict_loaded = al['labels']

# Contar delegaciones normales y anómalas por agente
subs = [e for e in raw_events if e['short_name'] == 'queue_subordinate_task']
all_agents = set()
for e in subs:
    for p in e['parties']:
        if p.startswith('Agent/'):
            all_agents.add(p)

metrics = {}
for agent in all_agents:
    person_id = agent.replace('Agent/', '')
    dept = person_dept.get(person_id, 'unknown')
    team = person_team.get(person_id, 'unknown')

    sent_normal   = 0
    sent_anom     = Counter()
    recv_normal   = 0
    recv_anom     = Counter()

    for e in subs:
        ags = [p for p in e['parties'] if p.startswith('Agent/')]
        if len(ags) < 2:
            continue
        src, tgt = ags[0], ags[1]
        label = label_dict_loaded.get(str(e['id']), 'normal')
        if src == agent:
            if label == 'normal': sent_normal += 1
            else: sent_anom[label] += 1
        if tgt == agent:
            if label == 'normal': recv_normal += 1
            else: recv_anom[label] += 1

    is_c2 = agent.replace('Agent/', '') in [
        'person:zoey_drydock', 'person:gabriel_sonar',
        'person:owen_hatch', 'person:evelyn_dock'
    ]
    incidents_involved = sorted(set(sent_anom) | set(recv_anom))

    metrics[agent] = {
        'agent_id':  agent,
        'person_id': person_id,
        'department': dept,
        'team':       team,
        'is_c2_agent': is_c2,
        'incidents_involved': incidents_involved,
        'sent_normal':   sent_normal,
        'sent_anomalous': dict(sent_anom),
        'recv_normal':   recv_normal,
        'recv_anomalous': dict(recv_anom),
        'total_sent':  sent_normal + sum(sent_anom.values()),
        'total_recv':  recv_normal + sum(recv_anom.values()),
        'anomaly_sent_total':  sum(sent_anom.values()),
        'anomaly_recv_total':  sum(recv_anom.values()),
    }

with open('../data/transf/agent_propagation_metrics.json', 'w') as f:
    json.dump({'agents': list(metrics.values())}, f, indent=2)

sorted_by_anom = sorted(metrics.values(), key=lambda x: x['anomaly_sent_total'] + x['anomaly_recv_total'], reverse=True)
print('Top agentes por participación anómala (sent+recv):')
for m in sorted_by_anom[:10]:
    name = m['person_id'].replace('person:', '')
    print(f'  {name:<22} dept:{m["department"].replace("department:",""):<30} sent_anóm:{m["anomaly_sent_total"]:3d}  recv_anóm:{m["anomaly_recv_total"]:3d}  c2:{m["is_c2_agent"]}')
print()
print('Saved → data/transf/agent_propagation_metrics.json')


Top agentes por participación anómala (sent+recv):
  evelyn_dock            dept:products                       sent_anóm: 21  recv_anóm: 21  c2:True
  gabriel_sonar          dept:information_technologies       sent_anóm: 21  recv_anóm: 20  c2:True
  zoey_drydock           dept:information_technologies       sent_anóm: 19  recv_anóm: 19  c2:True
  levi_signal            dept:information_technologies       sent_anóm: 18  recv_anóm: 18  c2:False
  victoria_rigging       dept:information_technologies       sent_anóm: 18  recv_anóm: 18  c2:False
  owen_hatch             dept:information_technologies       sent_anóm: 17  recv_anóm: 17  c2:True
  chloe_ballast          dept:information_technologies       sent_anóm: 16  recv_anóm: 16  c2:False
  mia_fender             dept:products                       sent_anóm: 15  recv_anóm: 15  c2:False
  olivia_keel            dept:products                       sent_anóm: 14  recv_anóm: 14  c2:False
  liam_anchor            dept:executive_suite        

#### 11. Intervention edges — aristas de delegación anotadas

> **Por qué:** para recomendar el punto de intervención óptimo hay que saber qué aristas del grafo de delegación aparecen en cadenas anómalas pero no en el flujo normal. La `anomaly_ratio` y el `intervention_score` cuantifican esto.

In [12]:
edge_counts = defaultdict(lambda: {'normal': 0, 'HiddenOrca': 0, 'MellowOtter': 0, 'SwiftWren': 0})

for e in subs:
    ags = [p for p in e['parties'] if p.startswith('Agent/')]
    if len(ags) < 2:
        continue
    src, tgt = ags[0], ags[1]
    label = label_dict_loaded.get(str(e['id']), 'normal')
    edge_counts[(src, tgt)][label] += 1

intervention_edges = []
for (src, tgt), counts in edge_counts.items():
    total_anom = counts['HiddenOrca'] + counts['MellowOtter'] + counts['SwiftWren']
    total_all  = counts['normal'] + total_anom
    anomaly_ratio = round(total_anom / total_all, 4) if total_all > 0 else 0
    # Score: más alto = mayor impacto de intervención
    # Penaliza si también tiene mucho tráfico normal (falsos positivos)
    incidents_present = sum(1 for k in ('HiddenOrca', 'MellowOtter', 'SwiftWren') if counts[k] > 0)
    intervention_score = round(anomaly_ratio * incidents_present, 4)

    intervention_edges.append({
        'from': src,
        'to':   tgt,
        'normal_count':     counts['normal'],
        'HiddenOrca_count': counts['HiddenOrca'],
        'MellowOtter_count': counts['MellowOtter'],
        'SwiftWren_count':  counts['SwiftWren'],
        'total_anomalous':  total_anom,
        'total_all':        total_all,
        'anomaly_ratio':    anomaly_ratio,
        'incidents_count':  incidents_present,
        'intervention_score': intervention_score,
        'in_all_incidents': incidents_present == 3,
    })

# Ordenar por score descendente
intervention_edges.sort(key=lambda x: x['intervention_score'], reverse=True)

with open('../data/transf/intervention_edges.json', 'w') as f:
    json.dump({'edges': intervention_edges}, f, indent=2)

top = [e for e in intervention_edges if e['intervention_score'] > 0][:15]
print(f'Aristas con actividad anómala: {len(top)}')
print(f'Aristas presentes en los 3 incidentes: {sum(1 for e in intervention_edges if e["in_all_incidents"])}')
print()
print('Top aristas por intervention_score:')
for e in top[:8]:
    src = e["from"].replace("Agent/person:","")
    tgt = e["to"].replace("Agent/person:","")
    print(f'  {src:<20} → {tgt:<20}  score:{e["intervention_score"]:.2f}  anóm:{e["total_anomalous"]:3d}  norm:{e["normal_count"]:3d}  incidentes:{e["incidents_count"]}')
print()
print('Saved → data/transf/intervention_edges.json')


Aristas con actividad anómala: 15
Aristas presentes en los 3 incidentes: 2

Top aristas por intervention_score:
  evelyn_dock          → gabriel_sonar         score:2.00  anóm:  4  norm:  2  incidentes:3
  zoey_drydock         → olivia_keel           score:2.00  anóm:  2  norm:  0  incidentes:2
  zoey_drydock         → zoey_drydock          score:2.00  anóm:  2  norm:  0  incidentes:2
  owen_hatch           → victoria_rigging      score:2.00  anóm:  3  norm:  0  incidentes:2
  olivia_keel          → owen_hatch            score:2.00  anóm:  3  norm:  0  incidentes:2
  liam_anchor          → michael_capstan       score:2.00  anóm:  2  norm:  0  incidentes:2
  chloe_ballast        → mia_fender            score:2.00  anóm:  5  norm:  0  incidentes:2
  levi_signal          → owen_hatch            score:2.00  anóm:  4  norm:  0  incidentes:2

Saved → data/transf/intervention_edges.json
